# Nagato NNUE — Live Training

Trains the HalfKP NNUE architecture on Nagato's 40-byte binary training data,
with live loss/accuracy plots and weight export in the `.bin` format the engine reads.

**Architecture:** `[FT:6400→256] → pairwise(128×2) → 4×[256→32→1] + PSQT[4]`  
**Loss:** `λ·MSE(σ(pred), σ(cp)) + (1−λ)·CE(σ(pred), wdl)`  where σ uses K=400

In [ ]:
%matplotlib widget
import struct, os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output
import ipywidgets as widgets
from tqdm.notebook import tqdm

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## Architecture Constants

In [ ]:
# ── Feature Transformer ──────────────────────────────────────────
KING_BUCKETS      = 10
PIECES_EX_KING    = 5
SQUARES           = 64
PER_COLOR_BUCKET  = PIECES_EX_KING * SQUARES    # 320
PER_BUCKET_FEATS  = PER_COLOR_BUCKET * 2        # 640
FT_SIZE           = KING_BUCKETS * PER_BUCKET_FEATS  # 6400

# ── Layer sizes ──────────────────────────────────────────────────
L1            = 256
L1_PAIR       = L1 // 2       # 128
L2_INPUT      = 2 * L1_PAIR   # 256  (pairwise stm + pairwise opp)
L2            = 32
NUM_STACKS    = 4             # piece-count-based layer stacks
SKIP          = 8             # skip connection from l2_in[0:8] → output
NUM_PSQT      = 4             # per-feature PSQT buckets

# ── Entry format ─────────────────────────────────────────────────
ENTRY_SIZE    = 40
SIGMOID_K     = 400.0

print(f"FT input: {FT_SIZE}  L1: {L1}  L2_INPUT: {L2_INPUT}  L2: {L2}  Stacks: {NUM_STACKS}")

## Feature Extraction from 40-byte Entries

In [ ]:
# Piece nibble → (piece_type 0-4, color 0/1)
# 1=wP 2=wN 3=wB 4=wR 5=wQ 6=wK  7=bP 8=bN 9=bB 10=bR 11=bQ 12=bK
NIBBLE_PIECE = {}  # nibble → (piece_type_idx, color)
for i, (pt, col) in enumerate([
    (0,0),(1,0),(2,0),(3,0),(4,0),(5,0),  # wP..wK  = nibbles 1-6
    (0,1),(1,1),(2,1),(3,1),(4,1),(5,1),  # bP..bK  = nibbles 7-12
]):
    NIBBLE_PIECE[i+1] = (pt, col)

def king_bucket_of(sq: int) -> int:
    """Mirrors Rust king_bucket_of."""
    file = sq & 7
    rank = sq >> 3
    fm = 7 - file if file >= 4 else file
    if 2 <= fm <= 3 and 2 <= rank <= 4: return 0
    if 1 <= fm <= 4 and 1 <= rank <= 6: return 1
    if fm >= 3 and 2 <= rank <= 5:      return 2
    if rank == 0:                        return 3
    if rank == 1:                        return 4
    if rank == 6:                        return 5
    if rank == 7:                        return 6
    if fm <= 1 and (rank <= 2 or rank >= 5): return 7
    if fm >= 4 or rank <= 0 or rank >= 7:   return 8
    return 9

def feat_white(piece_idx: int, color: int, sq: int, king_sq: int) -> int:
    """HalfKP feature index from White's king perspective."""
    bkt = king_bucket_of(king_sq)
    col_off = 0 if color == 0 else PER_COLOR_BUCKET
    return bkt * PER_BUCKET_FEATS + col_off + piece_idx * 64 + sq

def feat_black(piece_idx: int, color: int, sq: int, king_sq: int) -> int:
    """HalfKP feature index from Black's king perspective (board flipped)."""
    fsq  = sq ^ 56
    fksq = king_sq ^ 56
    bkt  = king_bucket_of(fksq)
    col_off = 0 if color == 1 else PER_COLOR_BUCKET
    return bkt * PER_BUCKET_FEATS + col_off + piece_idx * 64 + fsq

def extract_features(data: bytes):
    """Parse one 40-byte entry → (white_feats, black_feats, score, wdl, stm, piece_count)."""
    white_feats, black_feats = [], []
    wk, bk = 255, 255
    piece_list = []  # (piece_idx, color, sq)

    # First pass: find king squares
    for sq in range(64):
        byte_idx = sq >> 1
        nibble = (data[byte_idx] & 0x0F) if (sq & 1) == 0 else (data[byte_idx] >> 4)
        if nibble == 0: continue
        pt, col = NIBBLE_PIECE[nibble]
        if pt == 5:  # King
            if col == 0: wk = sq
            else:        bk = sq
        else:
            piece_list.append((pt, col, sq))

    # Second pass: compute features for non-king pieces
    for pt, col, sq in piece_list:
        white_feats.append(feat_white(pt, col, sq, wk))
        black_feats.append(feat_black(pt, col, sq, bk))

    side     = data[32]       # 0=White, 1=Black
    score    = struct.unpack_from('<h', data, 36)[0]  # cp from White's POV
    wdl_byte = data[38]
    wdl      = 1.0 if wdl_byte == 2 else (0.5 if wdl_byte == 1 else 0.0)
    pieces   = len(piece_list)

    return white_feats, black_feats, float(score), wdl, side, pieces

# Quick sanity check
import pathlib
bin_files = sorted(pathlib.Path('.').glob('*.bin'))
# find a good training file
TRAIN_FILE = next((str(f) for f in bin_files if 'train' in f.name), str(bin_files[0]) if bin_files else None)
print(f"Using: {TRAIN_FILE} ({os.path.getsize(TRAIN_FILE)//ENTRY_SIZE:,} entries)" if TRAIN_FILE else "No .bin found")

## Dataset

In [ ]:
class NagatoDataset(Dataset):
    def __init__(self, path: str, max_entries: int = None, shuffle: bool = True):
        with open(path, 'rb') as f:
            raw = f.read()
        n = len(raw) // ENTRY_SIZE
        self.entries = [raw[i*ENTRY_SIZE:(i+1)*ENTRY_SIZE] for i in range(n)]
        if shuffle:
            random.shuffle(self.entries)
        if max_entries:
            self.entries = self.entries[:max_entries]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        wf, bf, score, wdl, stm, pieces = extract_features(self.entries[idx])
        return {
            'white_feats': torch.tensor(wf, dtype=torch.long),
            'black_feats': torch.tensor(bf, dtype=torch.long),
            'score':  torch.tensor(score, dtype=torch.float32),
            'wdl':    torch.tensor(wdl,   dtype=torch.float32),
            'stm':    torch.tensor(stm,   dtype=torch.long),
            'pieces': torch.tensor(pieces,dtype=torch.long),
        }

def collate_fn(batch):
    """Pad variable-length feature lists and stack as sparse‐friendly batch."""
    B = len(batch)
    max_w = max(len(b['white_feats']) for b in batch)
    max_bk = max(len(b['black_feats']) for b in batch)
    wf = torch.zeros(B, max_w,  dtype=torch.long)
    bf = torch.zeros(B, max_bk, dtype=torch.long)
    wlen = torch.zeros(B, dtype=torch.long)
    blen = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        nw, nb = len(b['white_feats']), len(b['black_feats'])
        wf[i, :nw] = b['white_feats']
        bf[i, :nb] = b['black_feats']
        wlen[i] = nw; blen[i] = nb
    return {
        'white_feats': wf,  'white_len': wlen,
        'black_feats': bf,  'black_len': blen,
        'score':  torch.stack([b['score']  for b in batch]),
        'wdl':    torch.stack([b['wdl']    for b in batch]),
        'stm':    torch.stack([b['stm']    for b in batch]),
        'pieces': torch.stack([b['pieces'] for b in batch]),
    }

print("Dataset class defined.")

## Model Definition

In [ ]:
class NagatoNNUE(nn.Module):
    """
    HalfKP NNUE matching Nagato's Rust architecture:
      FT[6400→256] → pairwise-CReLU → 4×[256→32→1] + PSQT[6400→4]
    """
    def __init__(self):
        super().__init__()
        # Feature Transformer — shared between both perspectives
        self.ft_weight = nn.Parameter(torch.zeros(FT_SIZE, L1))
        self.ft_bias   = nn.Parameter(torch.zeros(L1))

        # PSQT — one scalar per (feature, bucket)
        self.psqt_weight = nn.Parameter(torch.zeros(FT_SIZE, NUM_PSQT))

        # Layer stacks — one per piece-count bucket
        self.l2_weight = nn.Parameter(torch.zeros(NUM_STACKS, L2_INPUT, L2))
        self.l2_bias   = nn.Parameter(torch.zeros(NUM_STACKS, L2))
        self.out_weight= nn.Parameter(torch.zeros(NUM_STACKS, L2))
        self.out_bias  = nn.Parameter(torch.zeros(NUM_STACKS))
        self.skip_weight=nn.Parameter(torch.zeros(NUM_STACKS, SKIP))

        self._init_weights()

    def _init_weights(self):
        nn.init.uniform_(self.ft_weight, -0.1, 0.1)
        nn.init.zeros_(self.ft_bias)
        nn.init.uniform_(self.l2_weight, -0.1, 0.1)
        nn.init.zeros_(self.l2_bias)
        nn.init.uniform_(self.out_weight, -0.05, 0.05)
        nn.init.zeros_(self.out_bias)
        nn.init.zeros_(self.psqt_weight)
        nn.init.zeros_(self.skip_weight)

    @staticmethod
    def psqt_bucket(pieces: torch.Tensor) -> torch.Tensor:
        """Map piece count → layer stack index (mirrors Rust psqt_bucket)."""
        idx = ((pieces.long().clamp(min=1) - 1) // 8).clamp(max=NUM_STACKS - 1)
        return idx

    def _accumulate(self, feats: torch.Tensor, flen: torch.Tensor) -> torch.Tensor:
        """
        Batch-sparse FT accumulation.
        feats: [B, max_f] indices, flen: [B] valid counts
        Returns: [B, L1] accumulated + bias
        """
        B, maxF = feats.shape
        # embed all (including padding zeros) then zero out padding
        acc = self.ft_bias.unsqueeze(0).expand(B, -1).clone()  # [B, L1]
        # scatter-add: for each sample, sum embeddings of valid features
        for i in range(B):
            n = flen[i].item()
            if n > 0:
                acc[i] += self.ft_weight[feats[i, :n]].sum(0)
        return acc

    def forward(self, batch):
        wf    = batch['white_feats'].to(DEVICE)   # [B, max_w]
        bf    = batch['black_feats'].to(DEVICE)   # [B, max_b]
        wlen  = batch['white_len'].to(DEVICE)     # [B]
        blen  = batch['black_len'].to(DEVICE)     # [B]
        stm   = batch['stm'].to(DEVICE)           # [B]  0=White 1=Black
        pcs   = batch['pieces'].to(DEVICE)        # [B]
        B     = wf.shape[0]

        l1_w = self._accumulate(wf, wlen)   # [B, L1]
        l1_b = self._accumulate(bf, blen)   # [B, L1]

        # Select stm / opp based on side to move
        stm_mask = stm.bool()  # True = Black is STM
        l1_stm = torch.where(stm_mask.unsqueeze(1), l1_b, l1_w)  # [B, L1]
        l1_opp = torch.where(stm_mask.unsqueeze(1), l1_w, l1_b)  # [B, L1]

        # Pairwise CReLU: crelu(x[0:128]) * crelu(x[128:256]) → [B, 128] each side
        def pairwise(x):
            a = x[:, :L1_PAIR].clamp(0, 1)
            b = x[:, L1_PAIR:].clamp(0, 1)
            return a * b

        l2_in = torch.cat([pairwise(l1_stm), pairwise(l1_opp)], dim=1)  # [B, 256]

        # Layer stacks (per piece-count bucket)
        stack_idx = self.psqt_bucket(pcs)  # [B]

        # Gather weights for each sample's stack
        l2_w  = self.l2_weight[stack_idx]   # [B, L2_INPUT, L2]
        l2_b  = self.l2_bias[stack_idx]     # [B, L2]
        out_w = self.out_weight[stack_idx]  # [B, L2]
        out_b = self.out_bias[stack_idx]    # [B]
        skip_w= self.skip_weight[stack_idx] # [B, SKIP]

        # L2 forward: [B, L2_INPUT] × [B, L2_INPUT, L2] → [B, L2], then CReLU
        l2_out = torch.bmm(l2_in.unsqueeze(1), l2_w).squeeze(1) + l2_b  # [B, L2]
        l2_out = l2_out.clamp(0, 1)

        # Output:  l2_out·out_w + out_b + l2_in[:SKIP]·skip_w
        positional = (l2_out * out_w).sum(1) + out_b
        positional = positional + (l2_in[:, :SKIP] * skip_w).sum(1)

        # PSQT
        psqt_stm_feats = torch.where(stm_mask.unsqueeze(1), bf, wf)   # [B, max]
        psqt_opp_feats = torch.where(stm_mask.unsqueeze(1), wf, bf)   # [B, max]
        psqt_stm_len   = torch.where(stm_mask, blen, wlen)
        psqt_opp_len   = torch.where(stm_mask, wlen, blen)

        def psqt_sum(feats, flen):
            ps = torch.zeros(B, device=DEVICE)
            for i in range(B):
                n = psqt_stm_len[i] if feats is psqt_stm_feats else psqt_opp_len[i]
                n = flen[i].item()
                if n > 0:
                    si = stack_idx[i].item()
                    ps[i] = self.psqt_weight[feats[i, :n], si].sum()
            return ps

        psqt_stm = psqt_sum(psqt_stm_feats, psqt_stm_len)  # [B]
        psqt_opp = psqt_sum(psqt_opp_feats, psqt_opp_len)  # [B]

        output = positional + psqt_stm - psqt_opp  # [B]
        return output

model = NagatoNNUE().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"  FT:    {FT_SIZE * L1:,}  ({FT_SIZE}×{L1})")
print(f"  PSQT:  {FT_SIZE * NUM_PSQT:,}  ({FT_SIZE}×{NUM_PSQT})")
print(f"  L2:    {NUM_STACKS * L2_INPUT * L2:,}  ({NUM_STACKS}×{L2_INPUT}×{L2})")

## Loss Function

In [ ]:
def sigmoid_k(x: torch.Tensor, k: float = SIGMOID_K) -> torch.Tensor:
    return torch.sigmoid(x / k)

def nagato_loss(pred: torch.Tensor, score: torch.Tensor, wdl: torch.Tensor, lam: float) -> torch.Tensor:
    """
    Matches Rust trainer::loss():
      λ·MSE(σ(pred), σ(target_cp)) + (1-λ)·BCE(σ(pred), wdl)
    """
    p     = sigmoid_k(pred)
    t_eval= sigmoid_k(score)
    mse   = (p - t_eval).pow(2).mean()
    # Binary cross-entropy: -(wdl*log(p) + (1-wdl)*log(1-p))
    bce   = F.binary_cross_entropy(p, wdl, reduction='mean')
    return lam * mse + (1 - lam) * bce

def wdl_accuracy(pred: torch.Tensor, wdl: torch.Tensor) -> float:
    """Fraction of positions where predicted outcome matches WDL label."""
    p = sigmoid_k(pred).detach().cpu()
    w = wdl.detach().cpu()
    # Classify: p>0.6 → win, p<0.4 → loss, else draw
    pred_cls = torch.where(p > 0.6, torch.ones_like(p),
                torch.where(p < 0.4, torch.zeros_like(p), torch.full_like(p, 0.5)))
    return (pred_cls == w).float().mean().item()

print("Loss function defined.")

## Training Configuration

Adjust hyperparameters below, then run the next cell to start training.

In [ ]:
# ── Hyperparameters (edit freely) ────────────────────────────────
TRAIN_FILE    = 'lichess_train_50k.bin'   # training data path
MAX_ENTRIES   = 500_000                   # cap entries for speed (None = all)
VAL_SPLIT     = 0.05                      # 5% validation
BATCH_SIZE    = 1024
EPOCHS        = 20
LR            = 3e-3
LR_SCHEDULE   = True                      # halve LR at 50% and 75% of epochs
LAMBDA        = 0.5                       # mix: 0=pure WDL, 1=pure CP
WEIGHT_DECAY  = 1e-5
PLOT_EVERY    = 50                        # batches between live plot updates
CHECKPOINT    = 'nn_pytorch.bin'          # output weight file

print(f"Config: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, λ={LAMBDA}")
n_total = os.path.getsize(TRAIN_FILE) // ENTRY_SIZE
n_use   = min(MAX_ENTRIES, n_total) if MAX_ENTRIES else n_total
n_val   = int(n_use * VAL_SPLIT)
n_train = n_use - n_val
batches_per_epoch = math.ceil(n_train / BATCH_SIZE)
print(f"Data: {n_train:,} train + {n_val:,} val  ({batches_per_epoch} batches/epoch)")

## Live Training Loop

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
print("Loading dataset...")
full_ds = NagatoDataset(TRAIN_FILE, max_entries=MAX_ENTRIES, shuffle=True)
val_ds  = torch.utils.data.Subset(full_ds, range(len(full_ds) - n_val, len(full_ds)))
trn_ds  = torch.utils.data.Subset(full_ds, range(len(full_ds) - n_val))

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)
print(f"Train: {len(trn_ds):,}  Val: {len(val_ds):,}")

# ── Model + Optimiser ─────────────────────────────────────────────
model = NagatoNNUE().to(DEVICE)
optimiser = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = None
if LR_SCHEDULE:
    milestones = [int(EPOCHS * 0.5), int(EPOCHS * 0.75)]
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimiser, milestones=milestones, gamma=0.5)

# ── Live plot setup ────────────────────────────────────────────────
fig = plt.figure(figsize=(13, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
ax_loss = fig.add_subplot(gs[0])
ax_wdl  = fig.add_subplot(gs[1])
ax_lr   = fig.add_subplot(gs[2])

for ax, title in [(ax_loss, 'Loss'), (ax_wdl, 'WDL Accuracy'), (ax_lr, 'Learning Rate')]:
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

trn_losses, val_losses, trn_accs, val_accs, lrs, epochs_done = [], [], [], [], [], []
live_x, live_y = [], []  # batch-level loss for smooth curve

(line_trn_l,) = ax_loss.plot([], [], 'b-o', ms=4, label='train')
(line_val_l,) = ax_loss.plot([], [], 'r-o', ms=4, label='val')
(line_trn_a,) = ax_wdl.plot([], [],  'b-o', ms=4, label='train')
(line_val_a,) = ax_wdl.plot([], [],  'r-o', ms=4, label='val')
(line_lr,)    = ax_lr.plot([], [],   'g-o', ms=4)
ax_loss.legend(); ax_wdl.legend()

status_out = widgets.Output()
display(status_out)
plt.tight_layout()
plt.show()

def update_plots():
    for line, data, ax in [
        (line_trn_l, (epochs_done, trn_losses), ax_loss),
        (line_val_l, (epochs_done, val_losses), ax_loss),
        (line_trn_a, (epochs_done, trn_accs),   ax_wdl),
        (line_val_a, (epochs_done, val_accs),    ax_wdl),
        (line_lr,    (epochs_done, lrs),          ax_lr),
    ]:
        line.set_data(data[0], data[1])
        ax.relim(); ax.autoscale_view()
    fig.canvas.draw_idle()

# ── Training ───────────────────────────────────────────────────────
best_val_loss = float('inf')
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, running_acc, n_batches = 0.0, 0.0, 0

    epoch_bar = tqdm(trn_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for step, batch in enumerate(epoch_bar):
        optimiser.zero_grad()
        pred  = model(batch)
        score = batch['score'].to(DEVICE)
        wdl   = batch['wdl'].to(DEVICE)
        loss  = nagato_loss(pred, score, wdl, LAMBDA)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()

        bl = loss.item()
        ba = wdl_accuracy(pred, wdl)
        running_loss += bl; running_acc += ba; n_batches += 1
        epoch_bar.set_postfix(loss=f'{bl:.4f}', acc=f'{ba:.2%}')

        if step % PLOT_EVERY == 0:
            live_x.append(epoch - 1 + step / len(trn_loader))
            live_y.append(bl)

    # Validation
    model.eval()
    v_loss, v_acc, v_n = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            pred  = model(batch)
            score = batch['score'].to(DEVICE)
            wdl   = batch['wdl'].to(DEVICE)
            v_loss += nagato_loss(pred, score, wdl, LAMBDA).item()
            v_acc  += wdl_accuracy(pred, wdl)
            v_n    += 1

    t_loss = running_loss / n_batches
    vl     = v_loss / max(v_n, 1)
    ta     = running_acc / n_batches
    va     = v_acc / max(v_n, 1)
    curr_lr= optimiser.param_groups[0]['lr']

    trn_losses.append(t_loss); val_losses.append(vl)
    trn_accs.append(ta);       val_accs.append(va)
    lrs.append(curr_lr);       epochs_done.append(epoch)

    # Save checkpoint if best
    if vl < best_val_loss:
        best_val_loss = vl

    if scheduler:
        scheduler.step()

    update_plots()
    with status_out:
        clear_output(wait=True)
        elapsed = time.time() - t0
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"train_loss={t_loss:.5f}  val_loss={vl:.5f}  "
              f"train_acc={ta:.2%}  val_acc={va:.2%}  "
              f"lr={curr_lr:.6f}  [{elapsed:.0f}s]")

print(f"\nTraining complete. Best val loss: {best_val_loss:.5f}")

## Export Weights → NAGT Binary

In [ ]:
def export_nagt(model: NagatoNNUE, path: str):
    """
    Write weights in the NAGT format Nagato reads at runtime.
    Matches Rust trainer::save_weights() + network::load_weights_from_file().

    Layout (all f32 LE):
      [0]   magic   'NAGT'
      [4]   version u32 = 3
      [8]   ft_w    [FT_SIZE × L1]
      [+]   ft_b    [L1]
      [+]   psqt_w  [FT_SIZE × NUM_PSQT]
      [+]   for each of NUM_STACKS:
               l2_w  [L2_INPUT × L2]
               l2_b  [L2]
               out_w [L2]
               out_b scalar
               skip_w[SKIP]
    """
    w = model.cpu()
    ft_w   = w.ft_weight.detach().numpy()     # [FT_SIZE, L1]
    ft_b   = w.ft_bias.detach().numpy()       # [L1]
    psqt   = w.psqt_weight.detach().numpy()   # [FT_SIZE, NUM_PSQT]
    l2_wts = w.l2_weight.detach().numpy()     # [NUM_STACKS, L2_INPUT, L2]
    l2_bs  = w.l2_bias.detach().numpy()       # [NUM_STACKS, L2]
    out_wts= w.out_weight.detach().numpy()    # [NUM_STACKS, L2]
    out_bs = w.out_bias.detach().numpy()      # [NUM_STACKS]
    skip_w = w.skip_weight.detach().numpy()   # [NUM_STACKS, SKIP]

    with open(path, 'wb') as f:
        f.write(b'NAGT')
        f.write(struct.pack('<I', 3))

        # ft_w: row-major [FT_SIZE, L1]
        f.write(ft_w.astype('<f4').tobytes())
        f.write(ft_b.astype('<f4').tobytes())

        # psqt_w: [FT_SIZE, NUM_PSQT]
        f.write(psqt.astype('<f4').tobytes())

        # per-stack: l2_w, l2_b, out_w, out_b (scalar), skip_w
        for s in range(NUM_STACKS):
            f.write(l2_wts[s].astype('<f4').tobytes())   # [L2_INPUT, L2]
            f.write(l2_bs[s].astype('<f4').tobytes())    # [L2]
            f.write(out_wts[s].astype('<f4').tobytes())  # [L2]
            f.write(struct.pack('<f', float(out_bs[s])))
            f.write(skip_w[s].astype('<f4').tobytes())   # [SKIP]

    size = os.path.getsize(path)
    print(f"Exported: {path}  ({size:,} bytes)")
    w = w.to(DEVICE)  # put back on device

export_nagt(model, CHECKPOINT)
print("Done — load with: nagato setoption name NNUEPath value", CHECKPOINT)

## Sample Position Evaluation

In [ ]:
def evaluate_fen(fen: str) -> float:
    """Return centipawn estimate from trained PyTorch model."""
    import chess
    board = chess.Board(fen)
    # Build small 40-byte entry for the position
    from pipeline import pack_board, encode_castling, PIECE_NIBBLE
    packed = pack_board(board)
    entry = bytearray(ENTRY_SIZE)
    entry[0:32]  = packed
    entry[32]    = 0 if board.turn == chess.WHITE else 1
    entry[33]    = encode_castling(board)
    entry[34]    = chess.square_file(board.ep_square) if board.ep_square else 255
    entry[36:38] = struct.pack('<h', 0)  # score unused for inference
    entry[38]    = 1  # wdl unused for inference

    wf, bf, _, _, stm, pieces = extract_features(bytes(entry))
    batch = collate_fn([{
        'white_feats': torch.tensor(wf, dtype=torch.long),
        'black_feats': torch.tensor(bf, dtype=torch.long),
        'score':  torch.tensor(0.0),
        'wdl':    torch.tensor(0.5),
        'stm':    torch.tensor(stm, dtype=torch.long),
        'pieces': torch.tensor(pieces, dtype=torch.long),
    }])
    model.eval()
    with torch.no_grad():
        pred = model(batch).item()
    return pred

test_positions = [
    ('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 'Start'),
    ('rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1', '1.e4'),
    ('8/8/8/8/8/8/PPPPPPPP/RNBQKBNR w KQ - 0 1', 'White extra pieces'),
    ('r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4', 'Italian'),
]

for fen, name in test_positions:
    try:
        cp = evaluate_fen(fen)
        win_pct = sigmoid_k(torch.tensor(cp)).item()
        print(f"  {name:25s}  cp={cp:+8.1f}  win%={win_pct:.1%}")
    except Exception as e:
        print(f"  {name}: error — {e}")